# Домашнее задание 4. Dockerfile по правилам

Автор: *Смирнова Анастасия*, РМ2, 15.04.2026

Это задание выполняется в рамках модуля 4.

> Чтобы получить максимальный балл, убедитесь, что ваш отчет содержит код, структура понятна, а в выводах вы объясняете свои решения.

Возмите любой типовой веб-сервис (например, из [ДЗ 1](https://colab.research.google.com/drive/1f3n_Ic5L1B5nSNwrc4THL7LJgXffB3qm?usp=sharing#scrollTo=EQRTmZPa2hi1)), чтобы использовать его в качестве контейнеризируемого сервиса.

В качестве контейнеризируемого сервиса возьмем CRUD сервиса из ДЗ_1.
Запишем его в `app.py` для дальнешей передачи в `Dockerfile`.

In [1]:
%%writefile app.py
import json
import re
import os
from http.server import HTTPServer, BaseHTTPRequestHandler

# In-memory database
items_db = {
    "1": {
        "id": "1",
        "name": "Антифриз EURO G11",
        "price": 1025,
        "discount": 11,
        "category": "антифриз"
    },
    "2": {
        "id": "2", 
        "name": "Антифриз Синтек MULTIFREEZE",
        "price": 250,
        "discount": 38,
        "category": "антифриз"
    }
}
next_id = 3

class SimpleCRUDHandler(BaseHTTPRequestHandler):
    
    def _send_json_response(self, status_code, data=None):
        self.send_response(status_code)
        self.send_header('Content-type', 'application/json; charset=utf-8')
        self.end_headers()
        if data is not None:
            self.wfile.write(json.dumps(data, ensure_ascii=False).encode('utf-8'))
    
    def _parse_json_body(self):
        content_length = int(self.headers.get('Content-Length', 0))
        if content_length == 0:
            return None
        body = self.rfile.read(content_length)
        try:
            return json.loads(body.decode('utf-8'))
        except json.JSONDecodeError:
            return None
    
    def do_GET(self):
        global items_db
        match_all = re.match(r'^/items/?$', self.path)
        match_single = re.match(r'^/items/([0-9]+)/?$', self.path)
        
        if match_all:
            self._send_json_response(200, list(items_db.values()))
        elif match_single:
            item_id = match_single.group(1)
            if item_id in items_db:
                self._send_json_response(200, items_db[item_id])
            else:
                self._send_json_response(404, {'detail': 'Item not found'})
        else:
            self._send_json_response(404, {'detail': 'Not Found'})
    
    def do_POST(self):
        match_path = re.match(r'^/items/?$', self.path)
        if match_path:
            data_to_post = self._parse_json_body()
            
            if data_to_post is None:
                self._send_json_response(400, {'detail': 'Invalid JSON'})
                return
            
            global items_db, next_id
            item_to_post = dict(data_to_post)
            item_to_post['id'] = str(next_id)
            items_db[str(next_id)] = item_to_post
            
            next_id += 1
            
            self._send_json_response(201, item_to_post)
        else:
            self._send_json_response(404, {'detail': 'Not Found'})
    
    def log_message(self, format, *args):
        print(f"{self.address_string()} - {format % args}")

def run_server(port=8000):
    port = int(os.environ.get('PORT', 8000))
    server_address = ('0.0.0.0', port)
    httpd = HTTPServer(server_address, SimpleCRUDHandler)
    print(f'Запускаем CRUD сервис на порту {port}...')
    print(f'Инициализрована БД с {len(items_db)} товарами')
    print(f'API эндпоинты:')
    print(f'  GET    /items     - Список всех товаров')
    print(f'  GET    /items/<id> - Получить товар по ID')
    print(f'  POST   /items     - Создать новый товар')
    httpd.serve_forever()

if __name__ == '__main__':
    run_server()

Overwriting app.py


Сразу зафиксируем зависимости в `requirements.txt`.

In [2]:
%%writefile requirements.txt
# Зависимости отсутствуют

Overwriting requirements.txt


In [3]:
# %%bash
# LATEST_HADOLINT_VERSION="2.12.0"
# HADOLINT_BINARY_URL="https://github.com/hadolint/hadolint/releases/download/v${LATEST_HADOLINT_VERSION}/hadolint-Linux-x86_64"
# wget -q "${HADOLINT_BINARY_URL}" -O hadolint
# chmod +x hadolint
# sudo mv hadolint /usr/local/bin/hadolint
# apt-get update -qq > /dev/null
# apt-get install -y yamllint > /dev/null

Добавим корректный код для MacOS. Устанавливаем hadolint через brew.

In [4]:
%%bash
brew install hadolint

✔︎ JSON API formula.jws.json
✔︎ JSON API cask.jws.json
To reinstall 2.14.0, run:
  brew reinstall hadolint


In [5]:
%%writefile .hadolint.yaml
ignored:
  - DL3000
  - SC1010

Overwriting .hadolint.yaml


In [ ]:
%%bash
brew install yamllint

### 1. Написать Dockerfile для ML-приложения

*Ожидаемый артефакт: Dockerfile в [ячейке](#scrollTo=qtkxMjZbmkre)*


In [ ]:
%%writefile Dockerfile

FROM python:3.9-slim AS builder

WORKDIR /app

# Копируем и устанавливаем зависимости
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app.py .

# Создаем непривилегированного пользователя
RUN useradd -m -u 1000 appuser && \
    chown -R appuser:appuser /app
USER appuser

ENV PORT=8000
EXPOSE 8000

CMD ["python", "app.py"]

In [ ]:
%%bash
hadolint --version

In [ ]:
%%bash
#hadolint --version
hadolint Dockerfile

На данном этапе создан файл Dockerfile:

- Зафиксирована версия python через `FROM python:3.10-slim`;
- Установлена рабочая директория через `WORKDIR`;
- Скопированы зависимости из `requirements.txt` через `COPY`;
- Выполнено кеширование pip зависимостей через `--no-cache-dir` (если requirements.txt не менялся);
- Выполнено копирование `app.py`.

Таким образом, кеширование pip зависимостей позволяет ускорить повторные сборки за счет того, что зависимостри будут копироваться повторно только в случае изменения `requirements.txt`.

-  На последнем этапе происходит создание непривелигированного пользователя

Далее мы переключаемся на него и все последующие команды будут выполняться от appuser, а не от root. Это необходимо для безопасности. Мы ограничеваем действия от супер пользователя.

Наконец наше приложение запускается на порту 8000.

Ячейка 

```bash
#hadolint --version
hadolint Dockerfile
```

не вывела предупреждений, следовательно Dockerfile написан корректно.

## 2. Использовать многостадийные сборки docker-образов для ML-приложения


*Ожидаемый артефакт: Dockerfile в [ячейке](#scrollTo=iWzu-Tj5mlp2)*


In [10]:
%%writefile Dockerfile
# Сборка зависимостей
FROM python:3.9-slim AS builder

WORKDIR /build
COPY requirements.txt .
RUN pip install --user --no-cache-dir -r requirements.txt

# Финальный образ
FROM python:3.9-slim

WORKDIR /app

# Копируем зависимости из стадии builder
COPY --from=builder /root/.local /home/appuser/.local

COPY app.py .

RUN useradd -m -u 1000 appuser && \
    chown -R appuser:appuser /app && \
    chown -R appuser:appuser /home/appuser

# Добавляем пакеты в PATH
ENV PATH=/home/appuser/.local/bin:$PATH

USER appuser
ENV PORT=8000
EXPOSE 8000

CMD ["python", "app.py"]

Overwriting Dockerfile


In [11]:
%%bash
hadolint Dockerfile

На данном этапе реализовали многостадийную сборку для нашего сервиса.

На первом этапе происходит сборка зависимостей через builder.
- Устанавливаем рабочу директорию `/build`;
- Повторяем шаги копирования `requirements.txt` и кеширования pip зависимостей.

На втором этапе происходит создание финального образа, где только копируются только готовые пакеты и код приложения.
- Директория  `/app`;
- Копируем зависимости с этапа `builder`;
- Создаем непривелигированного пользователя;
- Добавляем `path` для того, чтобы Python мог найти установленные на этапе `builder` пакеты.
- Наконец запускаем наше приложения на порту 8000.
  
Такой подход позволяет уменьшить размер финального образа за счёт исключения временных файлов и кеша pip, а также повышает безопасность, оставляя только необходимые для работы компоненты.
Проверка через `hadolint` предупреждение не вывела, наш Dockerfile написан корректно.

## 3. Настроить внешние и внутренние сети, тома хранения (volumes) и рестарт в docker-compose файле


*Ожидаемый артефакт: docker-compose.yaml в [ячейке](#scrollTo=wj4nRTO3VKF5)*


In [12]:
%%writefile docker-compose.yaml
---
services:
  crud-api:
    build:
      context: .
      dockerfile: Dockerfile
    container_name: crud-api-container
    ports:
      - "8000:8000"
    volumes:
      - app-data:/app/data
    networks:
      - internal
      - external
    restart: unless-stopped
    environment:
      - PYTHONUNBUFFERED=1

volumes:
  app-data:
    driver: local

networks:
  internal:
    internal: true
  external:

Overwriting docker-compose.yaml


In [13]:
%%bash
yamllint docker-compose.yaml

1) Сервис `crud-api` поднимается через созданный DockerFile. Имя созданного контейнера: `crud-api-container`;
2) Поднимаем сервис на порту 8000;
3) Монтируем именованный том `app-data` в папку `/app/data` внутри контейнера. Данный сервис хранит данные в памяти, однако по условию задания мы инициализиурем тома хранения;
4) Подключает сервис к двум сетям: internal (внутренняя сеть) и external (внешняя сеть).
5) Restart `unless-stopped` - контейнер всегда перезапускается при падении, кроме случая ручной остановки;
6) `PYTHONUNBUFFERED=1` позволяет записывать логи сразу в  `docker-compose logs`.

Блок `volumes` объявляет именованный том `app-data`. driver: local - хранится локально на хосте.

Блок `networks`: internal — с флагом internal: true (только для общения между контейнерами, недоступна извне) external— обычная.

Ячейка

```bash
yamllint docker-compose.yaml
```

не выдала предупреждений, следовательно `docker-compose` реализован корректно.

## 4. Настроить минимальные и максимальные границы памяти и ЦПУ в docker-compose файле


*Ожидаемый артефакт: docker-compose.yml в [ячейке](#scrollTo=X8tLvmVdmpZm)*



In [14]:
%%writefile docker-compose.yaml
---
services:
  crud-api:
    build:
      context: .
      dockerfile: Dockerfile
    container_name: crud-api-container
    ports:
      - "8000:8000"
    volumes:
      - app-data:/app/data
    networks:
      - internal
      - external
    restart: unless-stopped
    environment:
      - PYTHONUNBUFFERED=1
    deploy:
      resources:
        limits:
          memory: 256M
          cpus: '0.50'
        reservations:
          memory: 128M
          cpus: '0.25'

volumes:
  app-data:
    driver: local

networks:
  internal:
    internal: true
  external:

Overwriting docker-compose.yaml


In [15]:
%%bash
yamllint docker-compose.yaml

In [16]:
%%bash
# Запустить через docker-compose
docker-compose -f docker-compose.yaml up -d

 Network hw_4_deployment_mipt_internal Creating 
 Network hw_4_deployment_mipt_internal Created 
 Network hw_4_deployment_mipt_external Creating 
 Network hw_4_deployment_mipt_external Created 
 Container crud-api-container Creating 
 Container crud-api-container Created 
 Container crud-api-container Starting 
 Container crud-api-container Started 


In [18]:
%%bash
curl http://localhost:8000/items

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   241    0   241    0     0  65524      0 --:--:-- --:--:-- --:--:-- 80333


[{"id": "1", "name": "Антифриз EURO G11", "price": 1025, "discount": 11, "category": "антифриз"}, {"id": "2", "name": "Антифриз Синтек MULTIFREEZE", "price": 250, "discount": 38, "category": "антифриз"}]

In [19]:
%%bash
docker logs crud-api-container

Запускаем CRUD сервис на порту 8000...
Инициализрована БД с 2 товарами
API эндпоинты:
  GET    /items     - Список всех товаров
  GET    /items/<id> - Получить товар по ID
  POST   /items     - Создать новый товар
192.168.65.1 - "GET /items HTTP/1.1" 200 -


In [20]:
%%bash
docker-compose -f docker-compose.yaml down

 Container crud-api-container Stopping 
 Container crud-api-container Stopped 
 Container crud-api-container Removing 
 Container crud-api-container Removed 
 Network hw_4_deployment_mipt_internal Removing 
 Network hw_4_deployment_mipt_external Removing 
 Network hw_4_deployment_mipt_internal Removed 
 Network hw_4_deployment_mipt_external Removed 


На localhost сервис успешно запускается.

## 5. Написать базовый деплой сервиса в Kubernetes, используя YAML-файлы


*Ожидаемый артефакт: YAML-файл в [ячейке](#scrollTo=17btbwNmmqT2)*

In [21]:
%%writefile deploy.yaml
---
apiVersion: apps/v1
kind: Deployment
metadata:
  name: crud-api
  labels:
    app: crud-api
spec:
  replicas: 1
  selector:
    matchLabels:
      app: crud-api
  template:
    metadata:
      labels:
        app: crud-api
    spec:
      containers:
        - name: crud-api
          image: crud-api:latest
          imagePullPolicy: IfNotPresent
          ports:
            - containerPort: 8000
          resources:
            limits:
              cpu: "500m"
              memory: "256Mi"
            requests:
              cpu: "250m"
              memory: "128Mi"
          livenessProbe:
            httpGet:
              path: /items
              port: 8000
            initialDelaySeconds: 10
            periodSeconds: 10
          readinessProbe:
            httpGet:
              path: /items
              port: 8000
            initialDelaySeconds: 5
            periodSeconds: 5
---
apiVersion: v1
kind: Service
metadata:
  name: crud-api-service
spec:
  selector:
    app: crud-api
  ports:
    - protocol: TCP
      port: 80
      targetPort: 8000
  type: LoadBalancer

Overwriting deploy.yaml


In [22]:
%%bash
yamllint deploy.yaml

## 6. Итоговое оформление

В итоговых выводах дайте 5–8 предложений о своем опыте работы с инструментами модуля: что оказалось простым, что вызвало трудности, какие выводы сделали о применимости Docker/Kubernetes в реальных проектах.



# Вывод

В ходе выполнения домашнего задания были освоены базовые принципы контейнеризации веб-сервисов с использованием Docker и оркестрации с помощью Kubernetes.

**Что оказалось простым:** Написание базового Dockerfile и docker-compose файлов — синтаксис интуитивно понятен. Настройка сетей и томов также не вызвала затруднений после изучения документации. Деплой сервиса на Render также не вызвал затруднений. Воспользовалась [инструкцией](https://ru.hexlet.io/blog/posts/kak-deploit-prilozhenie-na-render-gayd-dlya-frontenderov-i-bekenderov).

**Что вызвало трудности:** Многостадийная сборка потребовала внимания к деталям: флаг --user для pip, копирование /root/.local, настройка PATH. На Mac с Apple Silicon возникли проблемы бинарной совместимости pandas с numpy, что было решено заменой на встроенные структуры Python.

Docker и Kubernetes - незаменимые инструменты в реальных проектах. Docker обеспечивает воспроизводимость окружения и упрощает деплой. Kubernetes предоставляет декларативное управление инфраструктурой, автоматическое масштабирование и самовосстановление. Освоение этих инструментов критически важно для современной разработки ML-сервисов.